<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/02_time_series/01_forecasting_quest_co2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌋 Quest · Time-series forecasting: the planet's breathing curve

**March 1958, Mauna Loa volcano, Hawaii, 3 400 m above the Pacific.** A young chemist, *Charles David Keeling*,
installs an instrument to measure the carbon dioxide (CO₂) in the air, far from cities and forests. Everyone expects a
flat line. What he discovers instead becomes one of the most famous charts in science, the **Keeling curve**:

- a relentless **upward trend** (we burn fossil fuels),
- a perfectly regular **yearly wave**: every Northern-hemisphere spring, leaves grow and plants inhale CO₂;
  every autumn they fall and the CO₂ comes back. *The planet breathes.*

**Your mission:** using measurements from 1958 to 1996, forecast the CO₂ level of the next five years (1997-2001),
then go further and predict today's level from 2001 data. This is **forecasting**: predicting future values of a
series from its past.

| You will practise | Concept |
|---|---|
| Resampling and filling gaps | regular time index |
| Splitting **in time** (never shuffle!) | train = past, test = future |
| Naive & seasonal-naive baselines | the bar every model must beat |
| Decomposition (STL) | trend + seasonality + residual |
| Exponential smoothing & SARIMA | classical forecasting models |
| MAE / MAPE, prediction intervals | measuring and communicating uncertainty |

🧩 Complete the `# TODO` cells, run the `check(...)` cells, reveal a solution if you are stuck for more than 10 minutes.

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")   # statsmodels is chatty about convergence; we keep the notebook readable

quest("The planet's breathing curve", chapter="Track 2 · Time series · Forecasting", xp=300)

## 1. Load Keeling's measurements 📥

The weekly measurements from 1958 to 2001 ship with `statsmodels` (no download needed).

In [ ]:
co2_weekly = sm.datasets.co2.load_pandas().data["co2"]
print(co2_weekly.head())
print(f"{len(co2_weekly)} weeks, {co2_weekly.isna().sum()} missing measurements (instrument breakdowns, bad weather…)")

A time series needs a **regular** index (one value per period) without holes, because most models assume
"the value of the previous period" exists.

### 🎯 Exercise 1: build a clean monthly series
Resample the weekly series to **month start** frequency (`"MS"`) using the mean, then fill the remaining gaps by
**linear interpolation** between neighbours.

In [ ]:
co2 = ...   # TODO: hint: co2_weekly.resample("MS").mean() then .interpolate()
co2.plot(title="The Keeling curve (monthly mean CO₂, ppm)", ylabel="CO₂ (ppm)"); plt.show()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
co2 = co2_weekly.resample("MS").mean().interpolate()
co2.plot(title="The Keeling curve (monthly mean CO₂, ppm)", ylabel="CO₂ (ppm)"); plt.show()
```

</details>

In [ ]:
check(isinstance(co2, pd.Series) and len(co2) == 526 and co2.isna().sum() == 0 and co2.index.freqstr == "MS",
      "Clean monthly series: 526 months from March 1958 to December 2001, no gaps.",
      "Expected 526 monthly values without NaN. Did you use resample('MS').mean() then interpolate()?")

**ppm** = parts per million: 315 ppm means 315 molecules of CO₂ for every million molecules of air.

## 2. Split in time ✂️

In the tabular track we shuffled rows before splitting. **Never do that with time series.** Shuffling would let the
model peek at the future (July 1999 in training, June 1999 in test) and give a wildly optimistic score. The test
set must be **after** the training set, exactly as in real life.

### 🎯 Exercise 2: train = up to December 1996, test = January 1997 onwards

In [ ]:
train = ...   # TODO: hint: co2[:"1996-12"]
test = ...    # TODO: hint: co2["1997-01":]
H = len(test)                 # forecast horizon: how many months we must predict
print(f"train: {train.index[0]:%Y-%m} → {train.index[-1]:%Y-%m} ({len(train)} months)")
print(f"test : {test.index[0]:%Y-%m} → {test.index[-1]:%Y-%m} (horizon H = {H} months)")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
train = co2[:"1996-12"]
test = co2["1997-01":]
H = len(test)
print(f"train: {train.index[0]:%Y-%m} → {train.index[-1]:%Y-%m} ({len(train)} months)")
print(f"test : {test.index[0]:%Y-%m} → {test.index[-1]:%Y-%m} (horizon H = {H} months)")
```

</details>

In [ ]:
check(len(test) == 60 and train.index.max() < test.index.min(),
      "Perfect: 5 years (60 months) to forecast, and the test is strictly after the training data.",
      "The test set should contain the 60 months of 1997-2001, all after the training months.")

## 3. Baselines: how good is "no intelligence"? 🧱

Two metrics, both "lower is better":
- **MAE** (mean absolute error): average error in ppm. Easy to explain: "we are off by 0.4 ppm on average".
- **MAPE** (mean absolute percentage error): average error in % of the true value, handy to compare series of
  different scales.

And two classic baselines:
- **Naive**: tomorrow = today. Repeat the last observed value for the whole horizon.
- **Seasonal naive**: next January = last January. Repeat the last observed **year**.

In [ ]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

lb = Leaderboard("co2", metric="MAE", higher_is_better=False)

naive = pd.Series(train.iloc[-1], index=test.index)
lb.add("Naive (last value)", mae(test, naive), MAPE=round(mape(test, naive), 3))

### 🎯 Exercise 3: the seasonal-naive forecast
Take the last 12 months of `train` and repeat them to cover the 60 months of the test set.

In [ ]:
last_year = train.iloc[-12:].to_numpy()
seasonal_naive = pd.Series(..., index=test.index)   # TODO: hint: np.tile(last_year, H // 12)
lb.add("Seasonal naive", mae(test, seasonal_naive), MAPE=round(mape(test, seasonal_naive), 3))

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
last_year = train.iloc[-12:].to_numpy()
seasonal_naive = pd.Series(np.tile(last_year, H // 12), index=test.index)
lb.add("Seasonal naive", mae(test, seasonal_naive), MAPE=round(mape(test, seasonal_naive), 3))
```

</details>

In [ ]:
check(len(seasonal_naive) == 60 and mae(test, seasonal_naive) < mae(test, naive),
      "Seasonal naive beats naive: it captures the yearly wave. But it ignores the trend, so its error grows each year.",
      "seasonal_naive should have 60 values built by repeating the last 12 training months.")
ax = train["1990":].plot(label="train"); test.plot(ax=ax, label="test (truth)")
naive.plot(ax=ax, ls="--", label="naive"); seasonal_naive.plot(ax=ax, ls="--", label="seasonal naive")
ax.set(title="Baselines vs reality", ylabel="CO₂ (ppm)"); ax.legend(); plt.show()

## 4. Decompose: trend + season + noise 🔬

**STL** (*Seasonal-Trend decomposition using Loess*, Cleveland et al., 1990) splits a series into three parts:

$$ y_t = \underbrace{T_t}_{\text{trend}} + \underbrace{S_t}_{\text{seasonality}} + \underbrace{R_t}_{\text{residual}} $$

Looking at them separately tells you what a model must capture.

In [ ]:
from statsmodels.tsa.seasonal import STL
stl = STL(train, period=12, robust=True).fit()
stl.plot(); plt.show()

### 🎯 Exercise 4: read the decomposition
1. By how many ppm per year does the **trend** grow over the last 10 training years (1987-1996)?
2. What is the **amplitude** of the seasonal wave (max − min of the seasonal component over the last year)?

In [ ]:
trend = stl.trend
growth_per_year = ...   # TODO: (trend at 1996-12 − trend at 1986-12) / 10   (hint: trend["1996-12"].iloc[0])
season_amplitude = ...  # TODO: hint: stl.seasonal[-12:].max() - stl.seasonal[-12:].min()
print(f"Trend growth: {growth_per_year:.2f} ppm/year · seasonal amplitude: {season_amplitude:.1f} ppm")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
trend = stl.trend
growth_per_year = (trend["1996-12"].iloc[0] - trend["1986-12"].iloc[0]) / 10
season_amplitude = stl.seasonal.iloc[-12:].max() - stl.seasonal.iloc[-12:].min()
print(f"Trend growth: {growth_per_year:.2f} ppm/year · seasonal amplitude: {season_amplitude:.1f} ppm")
```

</details>

In [ ]:
check(1.2 < growth_per_year < 1.8 and 5 < season_amplitude < 8,
      "≈1.5 ppm more CO₂ every year, and a ~6 ppm breath every year. Any good model must capture both.",
      "Growth should be around 1.5 ppm/year and amplitude around 6 ppm.")
checkpoint("co2/decomposition", xp=60)

That gives an idea for a smarter baseline: **seasonal naive + drift**: repeat last year, and add the yearly growth
for each year ahead. Simple, explainable, and surprisingly strong.

In [ ]:
years_ahead = np.repeat(np.arange(1, H // 12 + 1), 12)
snaive_drift = seasonal_naive + years_ahead * growth_per_year
lb.add("Seasonal naive + drift", mae(test, snaive_drift), MAPE=round(mape(test, snaive_drift), 3))

## 5. Exponential smoothing (Holt-Winters) 📉

Invented in the late 1950s by Holt and Winters (to forecast sales for the US Navy and businesses), exponential
smoothing keeps three running estimates: **level**, **trend** and **seasonal** effects, each updated with a weighted
average where recent observations count more ("exponentially decaying weights").

### 🎯 Exercise 5: fit Holt-Winters with additive trend and additive seasonality (period 12)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
hw_model = ...   # TODO: ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit()
hw_forecast = hw_model.forecast(H)
lb.add("Holt-Winters", mae(test, hw_forecast), MAPE=round(mape(test, hw_forecast), 3))

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from statsmodels.tsa.holtwinters import ExponentialSmoothing
hw_model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit()
hw_forecast = hw_model.forecast(H)
lb.add("Holt-Winters", mae(test, hw_forecast), MAPE=round(mape(test, hw_forecast), 3))
```

</details>

In [ ]:
check(mae(test, hw_forecast) < mae(test, seasonal_naive),
      "Holt-Winters captures both the trend and the season: far better than the baselines.",
      "Holt-Winters should beat seasonal naive. Did you set trend='add' and seasonal='add'?")

## 6. SARIMA, and why a forecast needs an interval 🎯

**ARIMA** (Box & Jenkins, 1970) models a series from its own past values (*AR*), its past errors (*MA*), after
**differencing** it (*I*) to remove the trend. **SARIMA** adds the same machinery at the seasonal lag (12 months).
The model `SARIMA(1,1,1)(0,1,1)₁₂` is a classic for monthly data like ours (Box and Jenkins even used it on airline
passengers, the famous "airline model").

A single number is never the whole story: a forecast should come with a **prediction interval**, a range that
should contain the truth, say, 95% of the time. Intervals get wider the further you look ahead.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
sarima = SARIMAX(train, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
fc = sarima.get_forecast(H)
sarima_forecast, interval = fc.predicted_mean, fc.conf_int(alpha=0.05)
lb.add("SARIMA(1,1,1)(0,1,1)12", mae(test, sarima_forecast), MAPE=round(mape(test, sarima_forecast), 3))

ax = train["1992":].plot(label="train", color="gray")
test.plot(ax=ax, label="truth", color="black")
sarima_forecast.plot(ax=ax, label="SARIMA forecast", color="tab:red")
ax.fill_between(interval.index, interval.iloc[:, 0], interval.iloc[:, 1], color="tab:red", alpha=0.2, label="95% interval")
ax.set(title="SARIMA forecast with its uncertainty", ylabel="CO₂ (ppm)"); ax.legend(); plt.show()

coverage = ((test >= interval.iloc[:, 0]) & (test <= interval.iloc[:, 1])).mean()
print(f"Share of real values inside the 95% interval: {coverage:.0%}")

### 🎯 Exercise 6: who wins?
Display the leaderboard and store the name of the best model.

In [ ]:
board = lb.show()
best_model_name = ...   # TODO: hint: lb.best()["model"]
board

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
board = lb.show()
best_model_name = lb.best()["model"]
board
```

</details>

In [ ]:
check(best_model_name in {"Holt-Winters", "SARIMA(1,1,1)(0,1,1)12", "Seasonal naive + drift"},
      f"The champion is {best_model_name}. The humble 'seasonal naive + drift' is at (or very near) the top: always try simple first!",
      "Look at the first row of the leaderboard.")
checkpoint("co2/models", xp=100)

## 7. 🔮 The ultimate test: from 2001 to today

Let's refit SARIMA on **all** the data (1958-2001) and forecast more than 20 years ahead. The real measurements at
Mauna Loa (published by NOAA) give an annual mean of about **424.6 ppm for 2024**. How close does a model that has
never seen the 21st century get?

In [ ]:
full_model = SARIMAX(co2, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
future = full_model.get_forecast(steps=12 * 23)             # Jan 2002 → Dec 2024
pred_2024 = future.predicted_mean["2024"].mean()
low_2024, high_2024 = future.conf_int(alpha=0.05).loc["2024"].mean()
print(f"🔮 Forecast for 2024 made with data up to 2001: {pred_2024:.1f} ppm (95% interval {low_2024:.0f}–{high_2024:.0f})")
print(f"🌍 Measured in 2024 at Mauna Loa: ≈ 424.6 ppm  →  error {pred_2024 - 424.6:+.1f} ppm")

ax = co2.plot(label="measured (1958-2001)", color="black")
future.predicted_mean.plot(ax=ax, label="SARIMA forecast", color="tab:red")
ci = future.conf_int(alpha=0.05)
ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color="tab:red", alpha=0.15)
ax.scatter(pd.Timestamp("2024-07-01"), 424.6, s=120, marker="*", color="gold", edgecolor="black", zorder=5, label="2024 measured")
ax.set(title="Forecasting 23 years ahead", ylabel="CO₂ (ppm)"); ax.legend(); plt.show()

The model underestimates by more than 15 ppm, and the real value even lands **outside** its 95% interval: since 2001
emissions **accelerated**, so the growth rate rose from ~1.5 to ~2.5 ppm per year. A statistical model extrapolates the past; it cannot know that the world will change. This is the deepest
lesson of forecasting: **the further the horizon, the more humble you must be** (look at that widening interval).

## 🏁 Quest complete!

You built the full forecasting loop: clean index → time split → baselines → decomposition → classical models →
intervals → long-horizon reality check.

### 🚀 Your turn (bonus XP)
1. Try `ExponentialSmoothing(..., damped_trend=True)`. Better or worse on the 5-year test? Why?
2. Change the split to train up to 1991 and forecast 10 years. Which model degrades the least?
3. Fit SARIMA on `np.log(train)` (multiplicative seasonality) and transform back with `np.exp`.
4. Real competitions with time series: Kaggle's [M5 Forecasting](https://www.kaggle.com/competitions/m5-forecasting-accuracy)
   and Zindi's air quality challenges. Next notebook: `02_forecasting_app_bike_demand.ipynb`, where machine learning
   enters the forecasting game.

In [ ]:
checkpoint("co2/complete", xp=140)
badge("Keeling's Apprentice", "🌋")
progress()